# EMBER 2018 malware detection — Colab experiment

This notebook runs the Phase 2 EMBER experiment entirely in Google Colab. It
downloads (or mounts) the official EMBER 2018 archive, vectorizes the raw
features, trains **LightGBM** and a **PyTorch MLP**, and evaluates both models
once on the official **time-based test split**.

**Run order:** `Runtime > Run all`. Expect roughly **25 GB of free disk**,
**8–12 GB of RAM**, a GPU runtime for the MLP, and **1–4 hours** for full
vectorization plus training. `SMOKE_TEST = True` vectorizes only a few
thousand rows and finishes in minutes; use it to check the pipeline before a
full run, then set it back to `False`.

**Evaluation policy**

- Validation is the last 10% of labeled training samples ordered by
  `appeared`, so it is later in time than the training rows.
- The official EMBER test split is evaluated **exactly once**, in the final
  evaluation cell. Do not tune on the test set.
- The deployment model is selected on **validation macro F1** only.

**License status:** the downloaded archive contains no license document, so
the dataset is recorded as **license not verified**. The EMBER repository
(`elastic/ember`) states that its data files are MIT-licensed and its source
code is AGPL-3.0. Do not redistribute the dataset. This is an academic
experiment, not a production security control.

The project does **not** extract EMBER locally. The local
`Downloads\ember_dataset_2018_2.tar.bz2` file is left untouched.

## 1. Configuration

In [ ]:
from pathlib import Path

CLASS_NAMES = ["Benign", "Malware"]

# "drive" uses an archive already uploaded to Google Drive (recommended).
# "download" fetches the official archive into /content with wget -c.
ARCHIVE_MODE = "drive"
DRIVE_ARCHIVE_PATH = Path("/content/drive/MyDrive/ember/ember_dataset_2018_2.tar.bz2")
DATASET_URL = "https://ember.elastic.co/ember_dataset_2018_2.tar.bz2"
DATASET_SHA256 = "b6052eb8d350a49a8d5a5396fbe7d16cf42848b86ff969b77464434cf2997812"
VERIFY_ARCHIVE_SHA256 = True

EXTRACT_DIR = Path("/content/ember_extract")
OUTPUT_DIR = Path("/content/ember_outputs")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/ember/outputs")

SMOKE_TEST = False
SMOKE_TRAIN_ROWS = 4000
SMOKE_TEST_ROWS = 1000

VALIDATION_FRACTION = 0.10
RANDOM_SEED = 42

LGB_NUM_BOOST_ROUND = 1200
LGB_EARLY_STOPPING = 50
LGB_NUM_LEAVES = 512
LGB_LEARNING_RATE = 0.05

MLP_MAX_EPOCHS = 10
MLP_PATIENCE = 3
MLP_BATCH_SIZE = 1024
MLP_LEARNING_RATE = 1e-3
MLP_NUM_WORKERS = 2
MLP_CPU_ROW_LIMIT = 200_000

IMPORTANCE_LIMIT = 15

## 2. Install dependencies

In [ ]:
import subprocess
import sys

def pip_install(*packages: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install("lightgbm>=4.5", "lief", "reportlab", "tqdm")
pip_install("--no-deps", "git+https://github.com/elastic/ember.git")

import ember
import lief
import lightgbm
import reportlab
import sklearn

extractor = ember.PEFeatureExtractor(feature_version=2, print_feature_warning=False)
print("EMBER feature dimension:", extractor.dim)
print("lightgbm:", lightgbm.__version__)
print("lief:", lief.__version__)
print("scikit-learn:", sklearn.__version__)

## 3. Obtain and extract the EMBER archive

**Drive mode (recommended).** Upload `ember_dataset_2018_2.tar.bz2` to
`MyDrive/ember/`. The archive is about 1.7 GB compressed. If your Drive path
differs, edit `DRIVE_ARCHIVE_PATH` above.

**Download mode.** Set `ARCHIVE_MODE = "download"`; the official host is used
and the SHA-256 checksum is verified.

In [ ]:
import hashlib
import shutil
import subprocess
import tarfile
from pathlib import Path

def sha256_file(path: Path, chunk_size: int = 4 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

if ARCHIVE_MODE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    archive_path = DRIVE_ARCHIVE_PATH
    if not archive_path.is_file():
        raise FileNotFoundError(
            f"EMBER archive not found at {archive_path}. Upload it to Drive "
            "or set ARCHIVE_MODE = 'download'."
        )
elif ARCHIVE_MODE == "download":
    archive_path = Path("/content/ember_dataset_2018_2.tar.bz2")
    if not archive_path.is_file():
        subprocess.check_call(["wget", "-c", DATASET_URL, "-O", str(archive_path)])
else:
    raise ValueError("ARCHIVE_MODE must be 'drive' or 'download'.")

free_gb = shutil.disk_usage("/content").free / 1e9
print(f"Free disk before extraction: {free_gb:.1f} GB")
already_extracted = EXTRACT_DIR.exists() and any(
    EXTRACT_DIR.rglob("test_features.jsonl")
)
minimum_free_gb = 15 if already_extracted else 25
if free_gb < minimum_free_gb:
    raise RuntimeError(
        f"At least {minimum_free_gb} GB of free disk is required. "
        "Delete stale extraction or vectorization outputs first."
    )

archive_sha256 = None
if VERIFY_ARCHIVE_SHA256:
    print("Checking archive SHA-256 (this can take a minute)...")
    archive_sha256 = sha256_file(archive_path)
    print("sha256:", archive_sha256)
    if archive_sha256 != DATASET_SHA256:
        raise RuntimeError(
            "Archive checksum mismatch. Delete the bad copy and download again."
        )

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
matches = list(EXTRACT_DIR.rglob("test_features.jsonl"))
if not matches:
    print("Extracting archive. This takes several minutes...")
    with tarfile.open(archive_path, "r:bz2") as archive:
        try:
            archive.extractall(EXTRACT_DIR, filter="data")
        except TypeError:
            archive.extractall(EXTRACT_DIR)
    matches = list(EXTRACT_DIR.rglob("test_features.jsonl"))
if not matches:
    raise RuntimeError("test_features.jsonl was not found after extraction.")

RAW_DATA_DIR = matches[0].parent
DATA_DIR = RAW_DATA_DIR
print("Raw feature directory:", RAW_DATA_DIR)
for name in ["train_features_0.jsonl", "test_features.jsonl"]:
    path = RAW_DATA_DIR / name
    print(name, f"{path.stat().st_size / 1e6:.1f} MB" if path.exists() else "MISSING")

drive_root = Path("/content/drive/MyDrive")
OUTPUT_DIR = DRIVE_OUTPUT_DIR if drive_root.exists() else Path("/content/ember_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Artifacts will be written to:", OUTPUT_DIR)

## 4. Vectorize raw features

The official `ember.create_vectorized_features` writes `X_train.dat`,
`y_train.dat`, `X_test.dat`, and `y_test.dat` as memory-mapped arrays. A
streaming metadata pass supplies `appeared` timestamps for the time-based
split; it takes a few minutes on the full archive but keeps memory low.

In `SMOKE_TEST` mode only a few thousand raw lines are vectorized into a
separate directory; the rest of the notebook runs unchanged.

In [ ]:
import csv
import json
import time

import ember

extractor = ember.PEFeatureExtractor(feature_version=2, print_feature_warning=False)
FEATURE_DIM = extractor.dim
print("EMBER feature dimension:", FEATURE_DIM)

RAW_TRAIN_FILES = [f"train_features_{index}.jsonl" for index in range(6)]
RAW_TEST_FILE = "test_features.jsonl"
META_FIELDS = ["sha256", "appeared", "label", "avclass", "subset"]


def count_lines(path):
    with path.open("r", encoding="utf-8") as handle:
        return sum(1 for _ in handle)


def write_metadata(output_path, train_paths, test_path):
    # Stream rows instead of collecting them, so peak memory stays small.
    temporary_path = output_path.with_name(output_path.name + ".tmp")
    with temporary_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=META_FIELDS)
        writer.writeheader()
        for subset, paths in (("train", train_paths), ("test", [test_path])):
            for path in paths:
                with path.open("r", encoding="utf-8") as source:
                    for line in source:
                        raw = json.loads(line)
                        writer.writerow(
                            {
                                "sha256": raw.get("sha256", ""),
                                "appeared": raw.get("appeared", ""),
                                "label": raw.get("label", ""),
                                "avclass": raw.get("avclass", ""),
                                "subset": subset,
                            }
                        )
    temporary_path.replace(output_path)


def sample_lines(source, destination, limit):
    total = count_lines(source)
    stride = max(1, total // max(limit, 1))
    written = 0
    with source.open("r", encoding="utf-8") as source_handle, destination.open(
        "w", encoding="utf-8"
    ) as target:
        for index, line in enumerate(source_handle):
            if index % stride == 0:
                target.write(line)
                written += 1
                if written >= limit:
                    break
    return written


if SMOKE_TEST:
    DATA_DIR = Path("/content/ember2018_smoke")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if not (DATA_DIR / "X_train.dat").exists():
        limit_per_file = max(1, SMOKE_TRAIN_ROWS // len(RAW_TRAIN_FILES))
        for name in RAW_TRAIN_FILES:
            sample_lines(RAW_DATA_DIR / name, DATA_DIR / name, limit_per_file)
        sample_lines(
            RAW_DATA_DIR / RAW_TEST_FILE,
            DATA_DIR / RAW_TEST_FILE,
            SMOKE_TEST_ROWS,
        )
        print(
            "Vectorizing smoke subset: "
            f"{sum(count_lines(DATA_DIR / name) for name in RAW_TRAIN_FILES):,} train / "
            f"{count_lines(DATA_DIR / RAW_TEST_FILE):,} test rows"
        )
        start = time.time()
        ember.create_vectorized_features(str(DATA_DIR), feature_version=2)
        print("Vectorization seconds:", round(time.time() - start, 1))
    print("SMOKE TEST data directory:", DATA_DIR)
else:
    DATA_DIR = RAW_DATA_DIR
    required = [
        DATA_DIR / "X_train.dat",
        DATA_DIR / "y_train.dat",
        DATA_DIR / "X_test.dat",
        DATA_DIR / "y_test.dat",
    ]
    if not all(path.exists() for path in required):
        start = time.time()
        ember.create_vectorized_features(str(DATA_DIR), feature_version=2)
        print("Vectorization seconds:", round(time.time() - start, 1))
    print("Vectorized data directory:", DATA_DIR)

# Row order matches the vectorized arrays because both follow raw-file order.
# A metadata.csv produced by ember.create_metadata is also accepted.
if not (DATA_DIR / "metadata.csv").exists():
    start = time.time()
    write_metadata(
        DATA_DIR / "metadata.csv",
        [DATA_DIR / name for name in RAW_TRAIN_FILES],
        DATA_DIR / RAW_TEST_FILE,
    )
    print("metadata.csv seconds:", round(time.time() - start, 1))

for name in ["X_train.dat", "y_train.dat", "X_test.dat", "y_test.dat", "metadata.csv"]:
    path = DATA_DIR / name
    print(name, f"{path.stat().st_size / 1e6:.1f} MB" if path.exists() else "MISSING")

## 5. Load the arrays and build a time-based split

EMBER training rows are stored in chronological order. The code below sorts
labeled rows by `appeared` anyway, writes a compact labeled training memmap,
and uses the last 10% (latest in time) as validation. The official test set is
untouched until the final evaluation cell.

In [ ]:
import numpy as np
import pandas as pd

n_train_all = int(np.memmap(DATA_DIR / "y_train.dat", dtype=np.float32, mode="r").shape[0])
n_test_all = int(np.memmap(DATA_DIR / "y_test.dat", dtype=np.float32, mode="r").shape[0])
X_train_all = np.memmap(
    DATA_DIR / "X_train.dat", dtype=np.float32, mode="r", shape=(n_train_all, FEATURE_DIM)
)
y_train_all = np.memmap(DATA_DIR / "y_train.dat", dtype=np.float32, mode="r")
X_test = np.memmap(
    DATA_DIR / "X_test.dat", dtype=np.float32, mode="r", shape=(n_test_all, FEATURE_DIM)
)
y_test_all = np.memmap(DATA_DIR / "y_test.dat", dtype=np.float32, mode="r")

metadata = pd.read_csv(DATA_DIR / "metadata.csv")
train_meta = metadata.loc[metadata["subset"] == "train"].reset_index(drop=True)
test_meta = metadata.loc[metadata["subset"] == "test"].reset_index(drop=True)
if len(train_meta) != n_train_all or len(test_meta) != n_test_all:
    raise RuntimeError("metadata.csv does not align with the vectorized arrays.")

labeled_mask = np.asarray(y_train_all) != -1
source_indices = np.flatnonzero(labeled_mask)

# Select labeled rows from the metadata first, then sort those rows by time.
appeared = None
if "appeared" in train_meta.columns:
    labeled_meta = train_meta.iloc[source_indices].reset_index(drop=True)
    parsed = pd.to_datetime(labeled_meta["appeared"], errors="coerce", utc=True)
    if parsed.notna().all():
        order = np.argsort(parsed.to_numpy(), kind="stable")
        source_indices = source_indices[order]
        appeared = parsed.iloc[order].reset_index(drop=True)
    else:
        print(
            "WARNING: some labeled rows have no 'appeared' value; "
            "keeping file order."
        )
else:
    print("WARNING: metadata has no 'appeared' column; keeping file order.")

n_labeled = int(len(source_indices))
TRAIN_X_PATH = DATA_DIR / "X_train_labeled.dat"
TRAIN_Y_PATH = DATA_DIR / "y_train_labeled.dat"
expected_x_bytes = n_labeled * FEATURE_DIM * np.dtype(np.float32).itemsize
expected_y_bytes = n_labeled * np.dtype(np.float32).itemsize
cache_is_valid = (
    TRAIN_X_PATH.exists()
    and TRAIN_Y_PATH.exists()
    and TRAIN_X_PATH.stat().st_size == expected_x_bytes
    and TRAIN_Y_PATH.stat().st_size == expected_y_bytes
)
if not cache_is_valid:
    print(f"Writing compact labeled training memmap: {n_labeled:,} rows")
    out_X = np.memmap(
        TRAIN_X_PATH, dtype=np.float32, mode="w+", shape=(n_labeled, FEATURE_DIM)
    )
    out_y = np.memmap(TRAIN_Y_PATH, dtype=np.float32, mode="w+", shape=n_labeled)
    chunk_size = 20_000
    for start in range(0, n_labeled, chunk_size):
        indices = source_indices[start : start + chunk_size]
        out_X[start : start + len(indices)] = np.asarray(X_train_all[indices])
        out_y[start : start + len(indices)] = np.asarray(y_train_all[indices])
    out_X.flush()
    out_y.flush()
    del out_X, out_y

X_train = np.memmap(
    TRAIN_X_PATH, dtype=np.float32, mode="r", shape=(n_labeled, FEATURE_DIM)
)
y_train = np.memmap(TRAIN_Y_PATH, dtype=np.float32, mode="r")
y_test = np.asarray(y_test_all).astype(np.int64)
test_eval_mask = y_test >= 0
y_train_int = np.asarray(y_train).astype(np.int64)

n_train = int(n_labeled * (1.0 - VALIDATION_FRACTION))
train_idx = np.arange(0, n_train)
validation_idx = np.arange(n_train, n_labeled)

print(f"Raw train rows: {n_train_all:,}; labeled: {n_labeled:,}")
print(f"Train: {len(train_idx):,}; validation: {len(validation_idx):,}")
print(f"Test: {n_test_all:,} (evaluated: {int(test_eval_mask.sum()):,})")
for name, y_values in [
    ("train", y_train_int[train_idx]),
    ("validation", y_train_int[validation_idx]),
    ("test", y_test[test_eval_mask]),
]:
    counts = np.bincount(y_values, minlength=2)
    print(f"{name} class counts: benign={counts[0]:,}, malware={counts[1]:,}")
    if counts.min() == 0:
        raise RuntimeError(f"The {name} split does not contain both classes.")

if appeared is not None:
    print("train time range:     ", appeared.iloc[0], "->", appeared.iloc[n_train - 1])
    print("validation time range:", appeared.iloc[n_train], "->", appeared.iloc[-1])
if "appeared" in test_meta.columns:
    test_appeared = pd.to_datetime(test_meta["appeared"], errors="coerce", utc=True)
    if test_appeared.notna().any():
        print("test time range:      ", test_appeared.min(), "->", test_appeared.max())
        if appeared is not None and test_appeared.min() <= appeared.iloc[-1]:
            print(
                "WARNING: test metadata starts before the last training timestamp; "
                "check the archive ordering."
            )

## 6. Metric helpers

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def binary_metrics(y_true, probabilities, threshold):
    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    predictions = (probabilities >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, predictions)),
        "precision_macro": float(
            precision_score(y_true, predictions, average="macro", zero_division=0)
        ),
        "recall_macro": float(
            recall_score(y_true, predictions, average="macro", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y_true, predictions, average="macro", zero_division=0)
        ),
        "f1_weighted": float(
            f1_score(y_true, predictions, average="weighted", zero_division=0)
        ),
        "roc_auc": float(roc_auc_score(y_true, probabilities)),
        "roc_auc_fpr_5e3": float(
            roc_auc_score(y_true, probabilities, max_fpr=5e-3)
        ),
        "confusion_matrix": confusion_matrix(y_true, predictions, labels=[0, 1]).tolist(),
        "threshold": float(threshold),
    }


def best_threshold(y_true, probabilities):
    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    thresholds = np.linspace(0.001, 0.999, 200)
    scores = [
        f1_score(
            y_true,
            (probabilities >= threshold).astype(int),
            average="macro",
            zero_division=0,
        )
        for threshold in thresholds
    ]
    best_index = int(np.argmax(scores))
    return float(thresholds[best_index]), float(scores[best_index])


def predict_proba_lgb(booster, features, batch_size=20_000):
    probabilities = np.empty(len(features), dtype=np.float64)
    for start in range(0, len(features), batch_size):
        batch = np.asarray(features[start : start + batch_size])
        probabilities[start : start + len(batch)] = booster.predict(batch)
    return probabilities

## 7. Train LightGBM

In [ ]:
import time

import lightgbm as lgb

params = {
    "objective": "binary",
    "metric": ["binary_logloss", "auc"],
    "boosting_type": "gbdt",
    "num_leaves": LGB_NUM_LEAVES,
    "learning_rate": LGB_LEARNING_RATE,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "min_data_in_leaf": 50,
    "num_threads": 0,
    "seed": RANDOM_SEED,
    "verbosity": -1,
}

# Rows are already ordered by time, so train/validation are contiguous
# slices. Fancy indexing here would copy gigabytes of features.
X_lgb_train = X_train[:n_train]
X_lgb_validation = X_train[n_train:]
dtrain = lgb.Dataset(
    X_lgb_train, label=y_train_int[:n_train], free_raw_data=True
)
dvalid = lgb.Dataset(
    X_lgb_validation,
    label=y_train_int[n_train:],
    reference=dtrain,
    free_raw_data=True,
)

start = time.time()
booster = lgb.train(
    params,
    dtrain,
    num_boost_round=LGB_NUM_BOOST_ROUND,
    valid_sets=[dvalid],
    callbacks=[
        lgb.early_stopping(LGB_EARLY_STOPPING, verbose=True),
        lgb.log_evaluation(50),
    ],
)
print(f"LightGBM training seconds: {time.time() - start:.1f}")
print("Best iteration:", booster.best_iteration)

validation_proba_lgb = predict_proba_lgb(booster, X_lgb_validation)
threshold_lgb, validation_f1_lgb = best_threshold(
    y_train_int[n_train:], validation_proba_lgb
)
validation_metrics_lgb = binary_metrics(
    y_train_int[n_train:], validation_proba_lgb, threshold_lgb
)
LGB_MODEL_PATH = OUTPUT_DIR / "ember_lightgbm.txt"
booster.save_model(str(LGB_MODEL_PATH), num_iteration=booster.best_iteration)
print("Validation macro F1:", round(validation_f1_lgb, 4), "at", threshold_lgb)
print("Saved:", LGB_MODEL_PATH)

## 8. Train the PyTorch MLP

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset


def compute_normalization(features, indices, chunk_size=10_000):
    dimension = features.shape[1]
    total = np.zeros(dimension, dtype=np.float64)
    total_squared = np.zeros(dimension, dtype=np.float64)
    for start in range(0, len(indices), chunk_size):
        batch = np.asarray(
            features[indices[start : start + chunk_size]], dtype=np.float64
        )
        total += batch.sum(axis=0)
        total_squared += np.square(batch).sum(axis=0)
    count = max(len(indices), 1)
    mean = total / count
    variance = np.maximum(total_squared / count - np.square(mean), 1e-12)
    return mean.astype(np.float32), np.sqrt(variance).astype(np.float32)


class EmberDataset(Dataset):
    def __init__(self, features, labels, indices, mean, scale):
        self.features = features
        self.labels = labels
        self.indices = np.asarray(indices)
        self.mean = mean
        self.scale = scale

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        row_index = int(self.indices[index])
        row = np.asarray(self.features[row_index], dtype=np.float32)
        row = (row - self.mean) / self.scale
        label = int(self.labels[row_index])
        return torch.from_numpy(row), torch.tensor(label, dtype=torch.long)


class EmberMLP(nn.Module):
    # Same layer layout and state-dict keys as malware_classifier.model.MalwareMLP.
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(64, output_dim),
        )

    def forward(self, features):
        return self.layers(features)


def mlp_probabilities(model, features, indices, mean, scale, device, batch_size=4096):
    model.eval()
    probabilities = np.empty(len(indices), dtype=np.float64)
    with torch.inference_mode():
        for start in range(0, len(indices), batch_size):
            batch_indices = indices[start : start + batch_size]
            batch = np.asarray(features[batch_indices], dtype=np.float32)
            batch = (batch - mean) / scale
            logits = model(torch.from_numpy(batch).to(device))
            probabilities[start : start + len(batch_indices)] = (
                torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            )
    return probabilities

In [ ]:
import os
import time

import torch

torch.manual_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

mlp_train_idx = train_idx
if not torch.cuda.is_available() and len(mlp_train_idx) > MLP_CPU_ROW_LIMIT:
    mlp_train_idx = mlp_train_idx[:MLP_CPU_ROW_LIMIT]
    print(
        f"No GPU: training the MLP on the first {MLP_CPU_ROW_LIMIT:,} "
        "chronological rows."
    )

mean, scale = compute_normalization(X_train, mlp_train_idx)
train_dataset = EmberDataset(X_train, y_train, mlp_train_idx, mean, scale)
validation_dataset = EmberDataset(
    X_train, y_train, validation_idx, mean, scale
)
worker_count = MLP_NUM_WORKERS if (os.cpu_count() or 1) > 2 else 0
train_loader = DataLoader(
    train_dataset,
    batch_size=MLP_BATCH_SIZE,
    shuffle=True,
    num_workers=worker_count,
    pin_memory=(device.type == "cuda"),
    drop_last=True,
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=4096,
    shuffle=False,
    num_workers=worker_count,
    pin_memory=(device.type == "cuda"),
)

model = EmberMLP(FEATURE_DIM, len(CLASS_NAMES)).to(device)
class_counts = np.bincount(
    y_train_int[mlp_train_idx], minlength=len(CLASS_NAMES)
)
class_weights = len(mlp_train_idx) / (
    len(CLASS_NAMES) * np.maximum(class_counts, 1)
)
loss_function = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32, device=device)
)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=MLP_LEARNING_RATE, weight_decay=1e-4
)

best_loss = float("inf")
best_state = None
epochs_without_improvement = 0
history = []
start = time.time()

for epoch in range(1, MLP_MAX_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    seen = 0
    for features, labels in train_loader:
        features = features.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_function(model(features), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        seen += len(labels)

    model.eval()
    validation_loss = 0.0
    validation_seen = 0
    with torch.inference_mode():
        for features, labels in validation_loader:
            features = features.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            batch_loss = loss_function(model(features), labels)
            validation_loss += batch_loss.item() * len(labels)
            validation_seen += len(labels)

    train_loss = total_loss / max(seen, 1)
    validation_loss = validation_loss / max(validation_seen, 1)
    history.append(
        {"epoch": epoch, "train_loss": train_loss, "validation_loss": validation_loss}
    )
    print(
        f"epoch {epoch:02d}: train_loss={train_loss:.4f}, "
        f"validation_loss={validation_loss:.4f}"
    )
    if validation_loss < best_loss:
        best_loss = validation_loss
        best_state = {
            name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= MLP_PATIENCE:
            print(f"Early stopping after epoch {epoch}.")
            break

if best_state is None:
    raise RuntimeError("MLP training did not produce a model.")
model.load_state_dict(best_state)
model.eval()
print(f"MLP training seconds: {time.time() - start:.1f}")

validation_proba_mlp = mlp_probabilities(
    model, X_train, validation_idx, mean, scale, device
)
threshold_mlp, validation_f1_mlp = best_threshold(
    y_train_int[validation_idx], validation_proba_mlp
)
validation_metrics_mlp = binary_metrics(
    y_train_int[validation_idx], validation_proba_mlp, threshold_mlp
)

MLP_MODEL_PATH = OUTPUT_DIR / "ember_mlp.pt"
feature_names = [f"ember_feature_{index:04d}" for index in range(FEATURE_DIM)]
# Mirrors the project checkpoint schema (see malware_classifier.inference),
# so load_checkpoint() can instantiate MalwareMLP from this file.
checkpoint = {
    "workflow": "ember2018",
    "dataset": "EMBER 2018 v2",
    "feature_names": feature_names,
    "class_names": CLASS_NAMES,
    "model_state_dict": best_state,
    "normalization_mean": mean.tolist(),
    "normalization_scale": scale.tolist(),
    "architecture": {
        "type": "MLP",
        "hidden_sizes": [128, 64],
        "activation": "ReLU",
        "dropout": 0.25,
    },
    "training_config": {
        "seed": RANDOM_SEED,
        "max_epochs": MLP_MAX_EPOCHS,
        "epochs_trained": len(history),
        "patience": MLP_PATIENCE,
        "batch_size": MLP_BATCH_SIZE,
        "optimizer": "AdamW",
        "learning_rate": MLP_LEARNING_RATE,
        "weight_decay": 1e-4,
        "best_validation_loss": best_loss,
        "device": str(device),
        "history": history,
    },
    "validation_metrics": validation_metrics_mlp,
    "threshold": threshold_mlp,
}
torch.save(checkpoint, MLP_MODEL_PATH)
print("Validation macro F1:", round(validation_f1_mlp, 4), "at", threshold_mlp)
print("Saved:", MLP_MODEL_PATH)

## 9. Final evaluation on the official time-based test split

This is the only cell that reads test labels. Both models are evaluated once,
at the thresholds selected on validation data.

In [ ]:
import json
from datetime import datetime, timezone

y_test_eval = y_test[test_eval_mask]
test_proba_lgb = predict_proba_lgb(booster, X_test)[test_eval_mask]
test_proba_mlp = mlp_probabilities(
    model, X_test, np.arange(n_test_all), mean, scale, device
)[test_eval_mask]

test_metrics_lgb = binary_metrics(y_test_eval, test_proba_lgb, threshold_lgb)
test_metrics_mlp = binary_metrics(y_test_eval, test_proba_mlp, threshold_mlp)

test_start = None
test_end = None
if "test_appeared" in globals() and test_appeared.notna().any():
    test_start = str(test_appeared.min())
    test_end = str(test_appeared.max())

if validation_metrics_lgb["f1_macro"] >= validation_metrics_mlp["f1_macro"]:
    deployment_model = "lightgbm"
else:
    deployment_model = "mlp"

importance = booster.feature_importance(importance_type="gain")
order = np.argsort(importance)[::-1][:IMPORTANCE_LIMIT]
top_features = [
    {"feature": f"ember_feature_{int(index):04d}", "gain": float(importance[index])}
    for index in order
]

import ember as ember_module
import lightgbm as lightgbm_module
import sklearn as sklearn_module
import torch as torch_module

metrics_payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "dataset": "EMBER 2018 v2",
    "dataset_family": "EMBER",
    "feature_version": 2,
    "feature_count": FEATURE_DIM,
    "license": "license not verified",
    "license_note": (
        "The downloaded archive contained no license document. The EMBER "
        "repository states that its data files are MIT-licensed and its source "
        "code is AGPL-3.0. The dataset is not redistributed."
    ),
    "archive": {
        "path": str(archive_path),
        "sha256": archive_sha256 or "not computed",
        "verified": bool(
            archive_sha256 is not None and archive_sha256 == DATASET_SHA256
        ),
    },
    "counts": {
        "train_raw": n_train_all,
        "train_labeled": n_labeled,
        "train": len(train_idx),
        "validation": len(validation_idx),
        "test": n_test_all,
        "test_evaluated": int(test_eval_mask.sum()),
    },
    "class_counts": {
        "train": np.bincount(y_train_int[train_idx], minlength=2).tolist(),
        "validation": np.bincount(
            y_train_int[validation_idx], minlength=2
        ).tolist(),
        "test": np.bincount(y_test_eval, minlength=2).tolist(),
    },
    "time_split": {
        "strategy": (
            "Labeled training rows sorted by 'appeared'; the latest 10% form "
            "validation. The official test split is evaluated once."
        ),
        "train_start": str(appeared.iloc[0]) if appeared is not None else None,
        "train_end": (
            str(appeared.iloc[n_train - 1]) if appeared is not None else None
        ),
        "validation_start": (
            str(appeared.iloc[n_train]) if appeared is not None else None
        ),
        "validation_end": (
            str(appeared.iloc[-1]) if appeared is not None else None
        ),
        "test_start": test_start,
        "test_end": test_end,
    },
    "seed": RANDOM_SEED,
    "validation_fraction": VALIDATION_FRACTION,
    "selection_metric": "validation_f1_macro",
    "deployment": {
        "model": deployment_model,
        "threshold": (
            threshold_lgb if deployment_model == "lightgbm" else threshold_mlp
        ),
    },
    "models": {
        "lightgbm": {
            "model_file": LGB_MODEL_PATH.name,
            "best_iteration": int(booster.best_iteration),
            "threshold": threshold_lgb,
            "validation": validation_metrics_lgb,
            "test": test_metrics_lgb,
            "top_gain_features": top_features,
        },
        "mlp": {
            "model_file": MLP_MODEL_PATH.name,
            "epochs_trained": len(history),
            "threshold": threshold_mlp,
            "validation": validation_metrics_mlp,
            "test": test_metrics_mlp,
        },
    },
    "environment": {
        "ember_feature_dimension": ember_module.PEFeatureExtractor(
            2, print_feature_warning=False
        ).dim,
        "lightgbm": lightgbm_module.__version__,
        "torch": torch_module.__version__,
        "scikit_learn": sklearn_module.__version__,
        "numpy": np.__version__,
        "device": str(device),
    },
}

METRICS_PATH = OUTPUT_DIR / "ember_metrics.json"
METRICS_PATH.write_text(
    json.dumps(metrics_payload, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("Validation macro F1: lightgbm=", round(validation_metrics_lgb["f1_macro"], 4),
      " mlp=", round(validation_metrics_mlp["f1_macro"], 4))
print("Deployment model selected on validation:", deployment_model)
for name, metrics in [("lightgbm", test_metrics_lgb), ("mlp", test_metrics_mlp)]:
    print(
        f"TEST {name}: accuracy={metrics['accuracy']:.4f}, "
        f"macro_f1={metrics['f1_macro']:.4f}, "
        f"roc_auc={metrics['roc_auc']:.4f}, "
        f"roc_auc_fpr_5e3={metrics['roc_auc_fpr_5e3']:.4f}, "
        f"threshold={metrics['threshold']:.2f}"
    )
print("Saved:", METRICS_PATH)

## 10. Build the model card and export the artifacts

In [ ]:
import html


def model_card_markdown(payload):
    models = payload["models"]
    lines = [
        "# EMBER 2018 malware-detection model card",
        "",
        "## Model details",
        "",
        f"- Dataset: {payload['dataset']} (feature version {payload['feature_version']})",
        f"- Features: {payload['feature_count']:,} pre-extracted PE features",
        "- Task: binary benign/malware classification",
        f"- Deployment model: {payload['deployment']['model']}",
        f"- Decision threshold: {payload['deployment']['threshold']:.2f}",
        f"- Generated: {payload['generated_at']}",
        "",
        "## Intended use",
        "",
        "Academic experiments and benchmark comparisons. The model classifies",
        "pre-extracted PE features; it does not execute files and is not a",
        "production security control.",
        "",
        "## Evaluation design",
        "",
        f"- {payload['time_split']['strategy']}",
        f"- Train time range: {payload['time_split']['train_start']} to {payload['time_split']['train_end']}",
        f"- Validation time range: {payload['time_split']['validation_start']} to {payload['time_split']['validation_end']}",
        f"- Test time range: {payload['time_split']['test_start']} to {payload['time_split']['test_end']}",
        f"- Train / validation / test rows: {payload['counts']['train']:,} / "
        f"{payload['counts']['validation']:,} / {payload['counts']['test_evaluated']:,}",
        f"- Selection metric: {payload['selection_metric']} (validation only)",
        "",
        "## Validation and test metrics",
        "",
        "| Model | Split | Accuracy | Macro F1 | ROC-AUC | ROC-AUC (FPR 5e-3) | Threshold |",
        "| --- | --- | --- | --- | --- | --- | --- |",
    ]
    for key in ["lightgbm", "mlp"]:
        for split in ["validation", "test"]:
            metrics = models[key][split]
            lines.append(
                f"| {key} | {split} | {metrics['accuracy']:.4f} | "
                f"{metrics['f1_macro']:.4f} | {metrics['roc_auc']:.4f} | "
                f"{metrics['roc_auc_fpr_5e3']:.4f} | {metrics['threshold']:.2f} |"
            )
    lines.extend(["", "### Test confusion matrices", ""])
    for key in ["lightgbm", "mlp"]:
        matrix = models[key]["test"]["confusion_matrix"]
        lines.append(f"{key} (rows = actual Benign/Malware, columns = predicted):")
        lines.append("")
        lines.append(f"- Benign: {matrix[0]}")
        lines.append(f"- Malware: {matrix[1]}")
        lines.append("")
    lines.extend(["## Top LightGBM gain features", ""])
    for item in models["lightgbm"]["top_gain_features"]:
        lines.append(f"- `{item['feature']}`: {item['gain']:.2f}")
    lines.extend(
        [
            "",
            "## Limitations",
            "",
            "- Features are static and can be evaded by packing, obfuscation, and",
            "  dynamic code loading.",
            "- The validation split is a single time-based hold-out; results depend",
            "  on the archive version and the vectorization environment.",
            "- The test split is evaluated once, but it is still a fixed benchmark.",
            "- The dataset archive contains no license document; status is",
            "  license not verified.",
            "",
            "## Safety",
            "",
            "Files are never executed. Dataset rows and models are not",
            "redistributed with the project repository.",
            "",
        ]
    )
    return "\n".join(lines)


def model_card_pdf(payload, output_path):
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import letter
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.lib.units import inch
    from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

    styles = getSampleStyleSheet()
    styles.add(
        ParagraphStyle(
            "CardTitle",
            parent=styles["Title"],
            fontName="Helvetica-Bold",
            fontSize=20,
            leading=25,
            textColor=colors.HexColor("#17365d"),
            spaceAfter=14,
        )
    )
    styles.add(
        ParagraphStyle(
            "CardHeading",
            parent=styles["Heading2"],
            fontName="Helvetica-Bold",
            fontSize=12,
            leading=16,
            textColor=colors.HexColor("#17365d"),
            spaceBefore=12,
            spaceAfter=5,
        )
    )
    styles.add(
        ParagraphStyle(
            "CardBody",
            parent=styles["BodyText"],
            fontName="Helvetica",
            fontSize=9,
            leading=13,
            spaceAfter=4,
        )
    )

    def body(text):
        return Paragraph(html.escape(text, quote=False), styles["CardBody"])

    story = [Paragraph("EMBER 2018 malware-detection model card", styles["CardTitle"])]
    details = [
        ("Dataset", payload["dataset"]),
        ("Features", f"{payload['feature_count']:,} pre-extracted PE features"),
        ("Deployment model", payload["deployment"]["model"]),
        ("Decision threshold", f"{payload['deployment']['threshold']:.2f}"),
        ("Generated", payload["generated_at"]),
        ("License", "license not verified"),
    ]
    for label, value in details:
        story.append(body(f"{label}: {value}"))

    story.append(Paragraph("Evaluation design", styles["CardHeading"]))
    for label, key in [
        ("Train range", "train_start"),
        ("Validation range", "validation_start"),
        ("Test range", "test_start"),
    ]:
        story.append(body(f"{label}: {payload['time_split'][key]}"))
    story.append(
        body(
            f"Rows: train {payload['counts']['train']:,}, validation "
            f"{payload['counts']['validation']:,}, test "
            f"{payload['counts']['test_evaluated']:,}. The official test split "
            "was evaluated once."
        )
    )

    story.append(Paragraph("Held-out metrics", styles["CardHeading"]))
    table_rows = [
        ["Model", "Split", "Accuracy", "Macro F1", "ROC-AUC", "Threshold"]
    ]
    for key in ["lightgbm", "mlp"]:
        for split in ["validation", "test"]:
            metrics = payload["models"][key][split]
            table_rows.append(
                [
                    key,
                    split,
                    f"{metrics['accuracy']:.4f}",
                    f"{metrics['f1_macro']:.4f}",
                    f"{metrics['roc_auc']:.4f}",
                    f"{metrics['threshold']:.2f}",
                ]
            )
    table = Table(
        table_rows,
        hAlign="LEFT",
        colWidths=[
            0.85 * inch,
            0.85 * inch,
            0.95 * inch,
            0.85 * inch,
            0.95 * inch,
            0.85 * inch,
        ],
    )
    table.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#17365d")),
                ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("FONTSIZE", (0, 0), (-1, -1), 8),
                ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#9fb3c8")),
                ("TOPPADDING", (0, 0), (-1, -1), 3),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
            ]
        )
    )
    story.append(table)

    story.append(Paragraph("Limitations", styles["CardHeading"]))
    for text in [
        "Static features can be evaded by packing, obfuscation, and dynamic loading.",
        "The validation split is a single time-based hold-out.",
        "The archive contained no license document; dataset status is license not verified.",
        "This model is not a production security control.",
    ]:
        story.append(body("- " + text))
    story.append(Spacer(1, 8))
    story.append(
        body(
            "Files are never executed. Dataset rows and trained models are not "
            "redistributed with the project repository."
        )
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    document = SimpleDocTemplate(
        str(output_path),
        pagesize=letter,
        leftMargin=0.75 * inch,
        rightMargin=0.75 * inch,
        topMargin=0.7 * inch,
        bottomMargin=0.7 * inch,
        title="EMBER 2018 model card",
    )
    document.build(story)

In [ ]:
import zipfile

CARD_MD_PATH = OUTPUT_DIR / "ember_model_card.md"
CARD_PDF_PATH = OUTPUT_DIR / "ember_model_card.pdf"
CARD_MD_PATH.write_text(model_card_markdown(metrics_payload), encoding="utf-8")
model_card_pdf(metrics_payload, CARD_PDF_PATH)
print("Wrote:", CARD_MD_PATH)
print("Wrote:", CARD_PDF_PATH)

ZIP_PATH = OUTPUT_DIR / "ember_experiment_artifacts.zip"
artifact_names = [
    "ember_lightgbm.txt",
    "ember_mlp.pt",
    "ember_metrics.json",
    "ember_model_card.md",
    "ember_model_card.pdf",
]
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zip_handle:
    for name in artifact_names:
        path = OUTPUT_DIR / name
        if path.exists():
            zip_handle.write(path, arcname=name)
print("Wrote:", ZIP_PATH)

try:
    from google.colab import files

    files.download(str(ZIP_PATH))
except Exception as error:
    print("Automatic download unavailable; download manually:", ZIP_PATH, error)

## Troubleshooting and notes

- **Archive not found:** upload `ember_dataset_2018_2.tar.bz2` to
  `MyDrive/ember/`, or set `ARCHIVE_MODE = "download"`.
- **Not enough disk:** full extraction plus vectorized arrays needs about
  25 GB free. Delete stale `/content/ember_extract` directories first.
- **Session disconnected:** `/content` is ephemeral. Re-run the archive and
  vectorization cells. Save artifacts to Drive (`OUTPUT_DIR` does this when
  Drive is mounted).
- **No GPU:** the MLP automatically limits itself to the first 200,000
  chronological training rows. LightGBM still uses the full data.
- **Out of memory during LightGBM:** lower `LGB_NUM_LEAVES` (for example to
  256) or switch to a high-RAM runtime. The vectorized arrays stay on disk and
  are memory-mapped.
- **Smoke test:** set `SMOKE_TEST = True` at the top to vectorize 4,000 train
  and 1,000 test rows. Results are not meaningful; the point is to verify the
  pipeline.
- **Licensing:** the dataset archive is marked “license not verified”. The
  EMBER repository states the data files are MIT-licensed; do not redistribute
  the data anyway.
- **Project integration:** this is a standalone Colab benchmark. The exported
  MLP checkpoint follows the project's checkpoint format, but the app's
  dashboard is not wired to EMBER features.

Full runs should use **Runtime > Change runtime type > GPU (T4)** and ideally
Colab Pro for the longer vectorization window.